In [ ]:
import numpy as np
import statsmodels.api as sm

In [2]:
import pandas as pd

In [14]:

# Column index -> desired name mapping from "Combined Monthly" sheet
# Row 2 contains short aliases (wti_chg_pct, etc.); row 0 is the master date
col_map = {
    0: "date",
    7: "wti_chg_pct",
    14: "vix_chg_pct",
    21: "sp500_chg_pct",
    31: "ng_chg_pct",
    39: "heatoil_chg_pct",
    47: "gld_chg_pct",
    57: "dxy_chg_pct",
    60: "tcmsy10_rate",
    64: "spr_inv_chg",
    67: "refin_util_rate",
    71: "crude_stk_excl_spr",
    75: "gpr_idx_chg",
    77: "gpr_threat_idx_chg",
    81: "us_crude_prod",
    85: "indpro",
    89: "igrea_chg_pct",
    97: "copper",
}

raw = pd.read_excel(
    "../data/data.xlsx",
    sheet_name="Combined Monthly",
    header=None,
    skiprows=3,          # skip the two header rows + alias row
    usecols=list(col_map.keys()),
)
raw.columns = [col_map[c] for c in raw.columns]
data = raw.set_index("date")
data.index = pd.to_datetime(data.index)
data = data.apply(pd.to_numeric, errors="coerce")


In [ ]:
# subset data from Dec 1990 to Dec 2025
subset_data = data.loc["1990-12-01":"2025-12-01"]
subset_data



,wti_chg_pct,vix_chg_pct,sp500_chg_pct,ng_chg_pct,heatoil_chg_pct,gld_chg_pct,dxy_chg_pct,tcmsy10_rate,spr_inv_chg,refin_util_rate,crude_stk_excl_spr,gpr_idx_chg,gpr_threat_idx_chg,us_crude_prod,indpro,igrea_chg_pct,copper
date,,,,,,,,,,,,,,,,,
1990-12-01,-0.028357,0.183490,0.024829,-0.207317,-0.040814,0.029358,-0.005388,0.0808,-0.003699,0.8325,-0.026476,-0.035158,-0.032471,-0.006633,-0.007340,-0.965634,0.072018
1991-01-01,-0.232633,-0.214500,0.041490,-0.281250,-0.156656,-0.067325,-0.016433,0.0803,-0.000139,0.8125,-0.038611,1.476128,0.849656,0.022077,-0.003256,-1.899457,-0.092473
1991-02-01,-0.123914,0.009030,0.067461,-0.005072,-0.057679,-0.006729,0.023120,0.0802,-0.002372,0.8400,0.020237,-0.105186,-0.295087,0.018267,-0.007389,32.853756,0.054374
1991-03-01,0.030446,-0.204899,0.022065,0.025547,0.068005,-0.024695,0.086294,0.0805,-0.014839,0.8300,0.019136,-0.562060,-0.472077,-0.011916,-0.005754,0.789833,-0.043518
1991-04-01,0.072123,0.047072,0.000533,-0.021352,0.015364,-0.003899,0.002409,0.0802,-0.012420,0.8375,0.014807,-0.342208,-0.247850,-0.004903,0.002639,-0.414185,-0.005676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-01,-0.077001,-0.117241,0.027513,-0.033226,-0.054951,0.027922,-0.022398,0.0423,0.002397,0.9576,-0.003443,0.033885,0.027578,0.007514,-0.002643,1.815317,0.024208
2025-09-01,-0.024707,0.006803,0.044825,0.111373,0.027863,0.100998,-0.000511,0.0416,0.005449,0.9315,-0.009909,-0.091602,-0.116006,0.001303,0.000426,0.407990,0.062718
2025-10-01,-0.023695,0.009259,0.026299,0.238067,0.047841,0.032307,0.020241,0.0411,0.006146,0.8786,0.007118,0.227564,0.184703,0.002603,-0.004529,-0.304549,0.048392


In [ ]:

# Multiple regression: WTI change % as dependent variable
target = "wti_chg_pct"
features = [c for c in subset_data.columns if c != target]

reg_data = data[[target] + features].dropna()

X = sm.add_constant(reg_data[features])
y = reg_data[target]

model = sm.OLS(y, X).fit()
print(model.summary())
